# LFM Instance Segmentation GFFT Workflow
This notebook trains a GFFT/Fourier-VQ MultiMAE Mask R-CNN instance segmentation model for crater detection. It loads a split crater instance dataset, builds the GFFT object-detection datamodule and TerraTorch task, runs fine-tuning, writes checkpoints, and creates validation prediction plots.

## Purpose of this notebook
Use this notebook as the active interactive GFFT instance-segmentation training workflow. The values in the **User Configuration** section mirror the most commonly changed command-line options; lower-level options stay on the centralized experiment-config defaults unless they are explicitly promoted into that section.

**Note**: the default configuration assumes single-band NAC data and a GFFT YAML with NAC normalization stats and backbone checkpoint metadata. For WAC data, change `DATASET_MODALITY` to `"wac"`, set `BAND_FILTER` to the WAC band indices you want to use, and point `GFFT_CONFIG_PATH` at the matching WAC GFFT YAML.


## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import sys

from functools import partialmethod
from glob import glob
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import seed_everything
from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, disable=False)

In [2]:
repo_root = Path.cwd().parent
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.utils import (
  create_timestamped_output_dir,
  plot_instance_cache_predictions,
  save_graha_instance_prediction_cache,
)
from lfm.all_models.inst_seg import build_gfft_notebook_configs
from lfm.full_model.inst_seg import instance_gfft_components

print("Successfully imported LFM modules")


/explore/nobackup/projects/lfm/lfm-full-env/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


Successfully imported LFM modules


## User Configuration

These are the values a notebook user is expected to edit for a normal GFFT instance-segmentation run.

#### Paths
`BASE_OUTPUT_DIR`: parent directory for timestamped notebook outputs. Checkpoints, config files, prediction caches, and plots are written under a new timestamped subdirectory.

`DATA_ROOT`: split dataset root. It should contain `train/`, `val/`, and `test/` folders, each with `chips/` and `labels/` subfolders.

`GFFT_CONFIG_PATH`: TerraTorch-style GFFT YAML. The notebook reads the backbone checkpoint path and pretraining normalization stats from this file.

`GFFT_BACKBONE_CHECKPOINT`: optional explicit GFFT backbone checkpoint override. Leave as `None` to use the checkpoint path from `GFFT_CONFIG_PATH`.

`LIGHTNING_CHECKPOINT`: optional GFFT Lightning checkpoint to resume from. Leave as `None` for a fresh fine-tune.

#### Data Selection
`DATASET_MODALITY`: dataset-level modality hint. `"nac"` resolves input mode and normalization to single-modality NAC behavior; `"wac"` resolves input mode to `"vis-uv"` and normalization modality to `"vis_uv"`.

`BAND_FILTER`: input band indices to keep. The default `[0]` uses a single NAC channel.

`MAX_TRAIN_SAMPLES`, `MAX_VAL_SAMPLES`, `MAX_TEST_SAMPLES`: optional split caps for quick experiments. Set any of these to `None` to use the full split.

#### Training
`BATCH_SIZE`: GFFT training batch size.

`NUM_WORKERS`: dataloader worker count.

`MAX_EPOCHS`: number of fine-tuning epochs.

`GFFT_BACKBONE_LR`, `GFFT_HEAD_LR`, `GFFT_LAYER_DECAY`, `GFFT_WEIGHT_DECAY`, `GFFT_WARMUP_STEPS`: optimizer schedule parameters passed into the GFFT object-detection task config.

#### Defaults Kept In Code
The notebook leaves these centralized defaults unchanged unless you add them to the config cell: `TARGET_SIZE=256`, `IMAGE_GLOB="*chip*.tif"`, `LABEL_GLOB="*label.*"`, `IMAGE_SUFFIX=None`, `LABEL_SUFFIX=None`, `GRAHA_STATS_BATCH_SIZE=16`, `GRAHA_VIS_UV_MERGE_METHOD="mean"`, `GRAHA_ANCHOR_SIZES=[[8], [16], [32], [64]]`, `GRAHA_ANCHOR_ASPECT_RATIOS=[0.5, 1.0, 2.0]`, `GRAHA_SCORE_THRESHOLD=0.5`, `PLOT_EVERY_N_EPOCHS=1`, `PLOT_N_SAMPLES=5`, `PREDICTION_SPLIT="val"`, `PREDICTION_N_SAMPLES=5`, `PREDICTION_SCORE_THRESHOLD=0.5`, `MASK_SHIFT=(0, 0)`, `IGNORE_NODATA_IN_LOSS=False`, `NODATA_IGNORE_INDEX=-1`, `SEED=42`.


In [ ]:
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_gfft_finetuning"
DATA_ROOT = "/explore/nobackup/projects/lfm/model_inputs/256_256_inputs/nac/nac_coco_inst_seg"
GFFT_CONFIG_PATH = repo_root / "graha-lunar-fm" / "terratorch_integration" / "configs" / "nac_craters" / "crater_detection_nac_only_fvqmultimae.yaml"
GFFT_BACKBONE_CHECKPOINT = None
LIGHTNING_CHECKPOINT = None

DATASET_MODALITY = "nac"
BAND_FILTER = [0]
MAX_TRAIN_SAMPLES = 500
MAX_VAL_SAMPLES = 500
MAX_TEST_SAMPLES = 500

BATCH_SIZE = 8
NUM_WORKERS = 10
MAX_EPOCHS = 1

GFFT_BACKBONE_LR = 5.0e-5
GFFT_HEAD_LR = 2.0e-4
GFFT_LAYER_DECAY = 0.75
GFFT_WEIGHT_DECAY = 0.05
GFFT_WARMUP_STEPS = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



The configuration cell above mirrors the active GFFT instance-segmentation training settings. Values not listed there use the centralized defaults documented in the previous markdown cell.


In [ ]:
OUTPUT_DIR = create_timestamped_output_dir(BASE_OUTPUT_DIR)

notebook_configs = build_gfft_notebook_configs(
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    base_output_dir=OUTPUT_DIR,
    gfft_config_path=GFFT_CONFIG_PATH,
    gfft_backbone_checkpoint=GFFT_BACKBONE_CHECKPOINT,
    graha_lightning_checkpoint=LIGHTNING_CHECKPOINT,
    dataset_modality=DATASET_MODALITY,
    max_epochs=MAX_EPOCHS,
    graha_batch_size=BATCH_SIZE,
    graha_num_workers=NUM_WORKERS,
    band_filter=BAND_FILTER,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    graha_backbone_lr=GFFT_BACKBONE_LR,
    graha_head_lr=GFFT_HEAD_LR,
    graha_layer_decay=GFFT_LAYER_DECAY,
    graha_weight_decay=GFFT_WEIGHT_DECAY,
    graha_warmup_steps=GFFT_WARMUP_STEPS,
)

config = notebook_configs.experiment_config
gfft_config = notebook_configs.gfft_config
deps = notebook_configs.dependencies

seed_everything(config.seed)
instance_gfft_components.save_config(gfft_config, OUTPUT_DIR)

print("Config created successfully")
print(f"Data root: {config.data_root}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"GFFT config YAML: {gfft_config.gfft_config_path}")
print(f"GFFT backbone weights: {gfft_config.backbone_weights}")
print(f"GFFT modality mode: {config.graha_input_modality_mode}")
print(f"Normalization modality: {config.normalization_modality}")


## Output Directory
The timestamped output directory was created while building the config. It contains the saved config, checkpoints, prediction caches, and plots for this run.


In [ ]:
print(f"Notebook output directory: {OUTPUT_DIR}")

## Create datamodule
1. Load GFFT pretraining stats from the configured YAML.
2. Create datamodule using pretraining stats and other config options.


In [ ]:
# STEP 1: pretraining stats
print("
STEP 1: Loading pretraining stats...")
print("="*60)

datamodule_cls = deps["GrahaObjectDetectionInstanceDataModule"]
means, stds = instance_gfft_components.get_normalization_stats(
    gfft_config,
    datamodule_cls,
)

print("Done.")


In [ ]:
print("
STEP 2: Creating datamodule and inspecting one training batch...")
print("=" * 60)

gfft_datamodule = instance_gfft_components.create_datamodule(
    gfft_config,
    datamodule_cls,
    means,
    stds,
)
gfft_sample_batch = instance_gfft_components.inspect_batch(gfft_datamodule)

print("Done.")


## Create Terratorch Task Object, Model


In [ ]:
task_cls = instance_gfft_components.make_downstream_object_detection_task_class(
    deps["LunarObjectDetectionTask"]
)

gfft_task = instance_gfft_components.create_task(
    gfft_config,
    task_cls,
    gfft_sample_batch,
)
instance_gfft_components.run_loss_smoke(gfft_task, gfft_sample_batch)


## Run Training

In [ ]:
trainer = instance_gfft_components.create_trainer(gfft_config, OUTPUT_DIR)
print(trainer)


In [ ]:
print("
" + "=" * 60)
print("Starting training.")
print("=" * 60)

ckpt_path = (
    str(gfft_config.lightning_checkpoint)
    if gfft_config.lightning_checkpoint is not None
    else None
)
trainer.fit(
    gfft_task,
    datamodule=gfft_datamodule,
    ckpt_path=ckpt_path,
)

print("Finished training.")


## Create And Display Validation Visualizations
Training writes checkpoint files during `trainer.fit()`. This section saves a small prediction cache on the configured split and renders the resulting plot in the notebook.


In [ ]:
prediction_cache = save_graha_instance_prediction_cache(
    task=gfft_task,
    datamodule=gfft_datamodule,
    output_dir=OUTPUT_DIR,
    model_name="gfft",
    split=config.prediction_split,
    n_samples=config.prediction_n_samples,
    score_threshold=config.prediction_score_threshold,
)

prediction_plot = plot_instance_cache_predictions(
    prediction_cache,
    OUTPUT_DIR / "plots" / "single_model" / "gfft_model",
    model_name="gfft",
    n_samples=config.prediction_n_samples,
    filename=f"{config.prediction_split}_instance_predictions.png",
)
print(f"Saved prediction plot: {prediction_plot}")


In [ ]:
img = mpimg.imread(prediction_plot)
plt.figure(figsize=(16, 14))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
del gfft_task, gfft_datamodule, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
